# Lab 1: Working with the OpenAI API and Reasoning Models

## Overview

This notebook demonstrates how to use the OpenAI API to build AI-powered applications. It walks through authentication, prompt design, response handling, and a practical comparison between standard chat models and reasoning models.

The example use case is a **career advisor**: a standard chat model first recommends a career path, then a reasoning model turns that recommendation into a detailed, step-by-step action plan.

## Objectives

- Configure the OpenAI API with proper authentication
- Design effective system and user prompts for a specific use case
- Process and render AI-generated responses
- Compare standard chat completions models with reasoning models

## Prerequisites

- An OpenAI API key set as the environment variable `OPENAI_API_KEY` (a `.env` file works well)
- The `openai`, `python-dotenv`, and `ipython` packages installed

## Step 1: Import Libraries and Load Environment Variables

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing. Set OPENAI_API_KEY in your environment or a .env file.")

## Step 2: Make Your First API Call

Initialize the OpenAI client and send a basic chat completion request to verify the setup.

In [ ]:
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(completion.choices[0].message.content)

### Understanding the Message List

The OpenAI API uses a structured message list to represent a conversation. Each message has a `role` — either `system`, `user`, or `assistant` — and a `content` field. Including prior turns in the list gives the model conversation context.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is the population of Paris?"},
    {"role": "assistant", "content": "As of 2021, the population of Paris is approximately 2.1 million people."}
]

## Step 3: Customize the Prompt

Craft a domain-specific system message and a structured user prompt to guide the model toward a targeted, useful response. Here the model acts as a career advisor and recommends a path based on a user profile.

In [ ]:
system_message = "You are a career advisor specializing in technology and data science fields. Your task is to help users find their ideal career path based on their skills, interests, and objectives."

user_prompt = """
Help me find my ideal career based on the following:

1. My Skillset
- Python programming
- Data science
- Machine learning

2. My current industry
- Technology

3. My interests
- Machine Learning in production
- Statistical analysis
- AI applications

4. My objectives
- Control over my hours
- Implementing AI solutions in real-world applications
- High value projects and impactful work

Now based on this, give me:
- A career suggestion
- Keep it concise and focused on the technology and data science fields.
- Reply in a friendly and optimistic tone.
"""

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]
)

chat_suggestion = completion.choices[0].message.content
display(Markdown(chat_suggestion))

## Step 4: Generate a Career Action Plan Using a Reasoning Model

Take the career suggestion from Step 3 and use a reasoning model (`o3-mini`) to produce a detailed, step-by-step action plan. Reasoning models are better suited for structured planning tasks that require multi-step logic.

The reasoning prompt below references the user profile and the suggested career, then requests three types of guidance: **skills to develop**, **resources to use**, and **milestones to achieve**.

In [ ]:
# Use the suggestion generated in Step 3. You can also paste a specific career here manually.
career_suggestion = chat_suggestion

reasoning_prompt = f"""
Here is the user's profile:
- Skills: Python, data science, machine learning
- Industry: Technology
- Interests: machine learning in production, statistical analysis, AI applications
- Objectives: control over working hours, real-world AI implementation, high-impact work

Based on this profile, the recommended career path is:
{career_suggestion}

Create a detailed, step-by-step plan for the user to pursue this career.
Structure your response with these three sections:
1. Skills to develop - the technical and soft skills to prioritize, in order.
2. Resources to use - specific courses, books, tools, or communities.
3. Milestones to achieve - measurable goals for the next 3, 6, and 12 months.

Be specific and actionable.
"""

completion = client.chat.completions.create(
    model="o3-mini",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": reasoning_prompt}
    ]
)

reasoning_response = completion.choices[0].message.content
display(Markdown(reasoning_response))

## Summary

In this lab we configured the OpenAI client, sent chat completion requests, structured multi-turn conversations with the message list, and designed a domain-specific prompt for a career advisor.

We then compared two approaches: a **standard chat model** (`gpt-4o-mini`) for a quick, conversational recommendation, and a **reasoning model** (`o3-mini`) for a structured, multi-step action plan. The reasoning model is the stronger choice when a task requires planning, ordering, and logical decomposition, while the chat model is faster and well-suited to direct, conversational responses.